# Best Perf + Consuption model

In [172]:
import os
import numpy as np
import random
from torchaudio.transforms import MelSpectrogram
from diffusers import ConfigMixin, Mel, ModelMixin
import torchaudio
import librosa
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from tqdm import tqdm
from prettytable import PrettyTable
from codecarbon import EmissionsTracker
from thop import profile, clever_format

np.random.seed(2026)

## Data preprocessing

Use the same data as last week

In [173]:
data_path = 'data'
     
data_folder = os.path.join(data_path, 'DESED_public_eval/audio/train/')
metadata_file = os.path.join(data_path, 'DESED_public_eval/metadata/train/public_weak.tsv')

print(f"data folder: {data_folder}")
print(f"data folder: {metadata_file}")

data folder: data/DESED_public_eval/audio/train/
data folder: data/DESED_public_eval/metadata/train/public_weak.tsv


In [174]:
from collections import OrderedDict

# Define the label-to-number mapping dictionary
classes_labels = OrderedDict(
    {
        "Alarm_bell_ringing": 0,
        "Blender": 1,
        "Cat": 2,
        "Dishes": 3,
        "Dog": 4,
        "Electric_shaver_toothbrush": 5,
        "Frying": 6,
        "Running_water": 7,
        "Speech": 8,
        "Vacuum_cleaner": 9,
    }
)

# Create a data loader
batch_size = 8
fs = 44100

target_len = 10 * fs

In [175]:
import pandas as pd

# Initialize an empty metadata dictionary
metadata = {}

# Read the metadata file (weak.tsv) while skipping the first line (headers)
with open(metadata_file, 'r') as file:
    next(file)  # Skip the first line (headers)
    for line in file:
        parts = line.strip().split()
        filename = parts[0]

        # Extract all event labels from the line
        event_labels = parts[1].split(',')

        # Map each event label to a number and store in a list
        encoded_event_labels = [classes_labels.get(label, -1) for label in event_labels]

        # Add the filename and encoded event labels to the metadata dictionary
        metadata[filename] = encoded_event_labels


In [176]:
# padding the audio
def pad_audio(waveform):

  if waveform.shape[-1] < target_len:
    waveform = torch.nn.functional.pad(
        waveform, (0, target_len - waveform.shape[-1]), mode="constant")

  elif len(waveform) > target_len:
    rand_onset = random.randint(0, len(waveform) - target_len)
    waveform = waveform[rand_onset:rand_onset + target_len]

  return waveform
     

In [177]:
def to_mono(waveform):
  if waveform.shape[0] > 1:
    indx = np.random.randint(0, waveform.shape[0] - 1)
    waveform = waveform[indx]
    waveform = waveform.unsqueeze(0)
  return waveform
    

In [178]:
class WeakDataset(Dataset):
    def __init__(self, data_folder, metadata, target_len, num_frames):
        self.data_folder = data_folder
        self.metadata = metadata
        self.file_list = os.listdir(data_folder)
        self.num_frames = num_frames
        self.mel = Mel(
            x_res=216,
            y_res=96,
            sample_rate=22050,
            n_fft=2048,
            hop_length=512,
            top_db=80,
        )
        self.target_len = target_len

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        filename = self.file_list[idx]
        file_path = os.path.join(self.data_folder, filename)
        self.mel.load_audio(file_path)
        mel_spec = [
                np.frombuffer(self.mel.audio_slice_to_image(slice).tobytes(), dtype="uint8").reshape(
                    (self.mel.y_res, self.mel.x_res)
                )
                / 255
            for slice in range(self.mel.get_number_of_slices())
            ]

        event_label = self.metadata[filename]
        #event_label = torch.tensor(event_label, dtype=torch.float32)
        mel_spec = torch.Tensor(mel_spec)

        return mel_spec, event_label[0]
     

## Training methods

In [179]:
def correct(output, target):
    predicted_digits = output.argmax(1)                            # pick digit with largest network output
    correct_ones = (predicted_digits == target).type(torch.float)  # 1.0 for correct, 0.0 for incorrect
    return correct_ones.sum().item()       

In [180]:
def train(data_loader, model, criterion, optimizer):
    # Use the GPU (if any)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.train()

    num_batches = len(data_loader)
    num_items = len(data_loader.dataset)

    total_loss = 0
    total_correct = 0
    for data, target in tqdm(data_loader, total=num_batches):
        # Copy data and targets to GPU
        data = data.to(device)
        target = target.to(device)

        # Do a forward pass
        output = model(data)

        # Calculate the loss
        loss = criterion(output, target)
        total_loss += loss

        # Count number of correct digits
        total_correct += correct(output, target)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    train_loss = total_loss/num_batches
    accuracy = total_correct/num_items
    print(f"Average loss: {train_loss:7f}, accuracy: {accuracy:.2%}")
    return train_loss.item(), accuracy

In [181]:
def test(data_loader, model, criterion, optimizer):
    # Use the GPU (if any)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.train()

    num_batches = len(data_loader)
    num_items = len(data_loader.dataset)

    total_loss = 0
    total_correct = 0
    for data, target in tqdm(data_loader, total=num_batches):
        # Copy data and targets to GPU
        data = data.to(device)
        target = target.to(device)
        with torch.no_grad():
            # Do a forward pass
            output = model(data)

        # Calculate the loss
        loss = criterion(output, target)
        total_loss += loss

        # Count number of correct digits
        total_correct += correct(output, target)


    test_loss = total_loss/num_batches
    accuracy = total_correct/num_items
    print(f"Average loss: {train_loss:7f}, accuracy: {accuracy:.2%}")
    return test_loss.item(), accuracy

In [182]:
num_frames = 64
dataset = WeakDataset(data_folder, metadata, target_len, num_frames)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [183]:
examples = enumerate(data_loader)
batch_idx, (example_data, example_targets) = next(examples)
example_data.shape


torch.Size([8, 1, 96, 216])

In [184]:
example_data.shape

torch.Size([8, 1, 96, 216])

In [185]:
file_list = os.listdir(data_folder)
audio_paths = []
for file in file_list:
    file_path = os.path.join(data_folder, filename)
    audio_paths.append(file_path)


In [186]:

audio_encoder.encode(audio_paths[:5])

tensor([[ 23.6139,  27.8850, -33.3395,  39.8689, -53.3891, -14.7269,  12.7873,
           8.4240, -43.1825,  23.9899,   9.0351,  15.6986,  -8.0012,  24.5727,
          34.3392,  55.5486,  29.7488,   3.4611, -40.3352, -33.3715,   3.0961,
          19.0516,  -1.1226,  34.1737, -57.5524, 100.0306, -28.6628,  70.0621,
          48.0845,  -4.5062,  -6.9211,  20.4688, -11.2932,  55.5967,   1.0866,
          14.8796,  93.2568, -79.0769,  -5.4812,  53.4737,  17.1708, -35.7934,
          33.0243,  -5.4372,  11.5652,   1.4604,  21.6186, -50.3697,  72.6198,
          -3.8549, -26.1101, -14.1557,   0.3964, -89.4671,  -4.9492, -50.6028,
          -3.9100, -43.9750,   5.1975,  49.8247, -14.4314,  14.2515, -38.5774,
          14.4030,  -0.3727, -26.1027, -45.4425,  12.1121, -13.9694,  -3.0818,
         -28.1804,  43.4458, -11.3017, -35.8989,   6.8429,  19.9560,  45.4046,
          10.3028,  23.2837,   7.8418,  15.5795, -50.9390,   3.4484,  20.0743,
          44.6632, -12.7075,   8.3422,  17.3191,  -2

In [187]:
print(example_data.shape)
audio_encoder(example_data)

torch.Size([8, 1, 96, 216])


tensor([[ 4.2555e+01,  6.5924e+01, -6.8066e+01,  7.9764e+01, -8.3484e+01,
          5.8439e-01,  1.0087e+01,  1.8787e+01, -7.3167e+01,  2.4810e+01,
          1.2972e+01,  4.9084e+01, -9.9149e+00,  9.6447e+01,  4.5737e+01,
          8.9809e+01,  4.1983e+01,  1.1830e+01, -5.4682e+01, -5.9489e+01,
          1.2152e+00,  2.4449e+01, -3.5068e+00,  9.1492e+01, -1.0015e+02,
          1.9217e+02, -8.4562e+01,  1.1127e+02,  1.1869e+02,  2.3046e+01,
          1.8250e+01,  2.5822e+01, -6.5713e+00,  8.8846e+01, -5.4463e-01,
          3.1939e+01,  1.9125e+02, -1.0775e+02, -1.4145e+01,  1.0968e+02,
          3.1134e+01, -6.0323e+01,  5.1267e+01, -2.6648e+01,  1.2872e+01,
          2.0381e+01,  5.4431e+01, -1.0231e+02,  1.4115e+02, -6.4860e+00,
         -2.7812e+01, -2.4144e+01,  6.8094e+00, -1.9081e+02, -2.4088e+01,
         -1.0288e+02,  1.6731e+01, -6.9290e+01,  1.9520e+01,  8.2651e+01,
         -5.3576e+01, -5.2442e+00, -6.0641e+01,  6.1824e+01, -1.9953e+01,
         -6.8259e+01, -7.2615e+01,  2.

In [188]:
class AudioClassifier(nn.Module):

    def __init__(self, n_classes, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.encoder = AudioEncoder.from_pretrained("teticio/audio-encoder")
        for param in self.encoder.parameters():
            param.requires_grad = False

        self.fc = nn.Linear(100, n_classes)

    def forward(self, X):
        encoded = self.encoder(X)
        return self.fc(encoded)

    

In [189]:
audio_classifier = AudioClassifier(n_classes=len(classes_labels))

An error occurred while trying to fetch teticio/audio-encoder: teticio/audio-encoder does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


In [190]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(audio_classifier.parameters(), lr=0.001)

In [192]:
%%time

# code carbon tracker
tracker_code_carbon = EmissionsTracker(tracking_mode="process", log_level="warning",output_dir=data_path)
tracker_code_carbon.start()

losses = []
accuracies = []
epochs = 100
for epoch in range(epochs):
    print(f"Training epoch: {epoch+1}")
    loss, acc = train(data_loader, audio_classifier, criterion, optimizer)
    losses.append(loss)
    accuracies.append(acc)
    
tracker_code_carbon.stop()

print('Training complete.')

[codecarbon WARNING @ 16:31:37] Multiple instances of codecarbon are allowed to run at the same time.


Training epoch: 1


 72%|███████▏  | 57/79 [00:16<00:06,  3.48it/s]

CPU times: user 2min 55s, sys: 765 ms, total: 2min 56s
Wall time: 16.4 s


RuntimeError: stack expects each tensor to be equal size, but got [2, 96, 216] at entry 0 and [1, 96, 216] at entry 1